In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import recall_score
from catboost import CatBoostClassifier, Pool

In [ ]:
data = pd.read_csv('./data/catalase_data.csv')

In [ ]:
nanozyme_year = pd.read_excel('./data/catalase.xlsx')

In [ ]:
positive = data[data['label'] == 1]
negative = data[data['label'] == 0]

In [ ]:
features = [
    'pearson',
    'space',
    'volume',
    'density',
    
    'band_gap',
    'efermi',
    'formation_energy_per_atom',
    
    'MagpieData mean NsValence',
    'MagpieData mean NpValence',
    'MagpieData mean NdValence',
    'MagpieData mean NfValence',
    
    'MagpieData mean NUnfilled',
    
    'MagpieData mean CovalentRadius',
    'MagpieData mean AtomicWeight',
    
    'MagpieData mean Electronegativity',
    'MagpieData mean GSmagmom',
    
    'MagpieData mean Row',
    'MagpieData mean Column'
]

In [ ]:
YEARS = list(range(2015, 2026))
N_RUNS = 500
ITERATIONS = 300
LEARNING_RATE = 0.05
DEPTH = 8
LOSS_FUNCTION = 'Logloss'
EVAL_METRIC = 'AUC'
RANDOM_SEED = 0
VERBOSE = 0

all_models = []

for YEAR in YEARS:
    pos_before = positive[positive['formula'].isin(
        nanozyme_year[nanozyme_year['year'] < YEAR].formula.tolist()
    )].reset_index(drop=True)

    test_positive = pos_before.sample(n=len(pos_before)//2, random_state=42).reset_index(drop=True)
    remove_test_positive = pos_before.drop(test_positive.index).reset_index(drop=True)

    X_test_positive = test_positive[features]
    y_test_positive = test_positive['label'].values

    n_positive = remove_test_positive.shape[0]
    n_negative = n_positive

    categorical_features = (
        ['pearson'] if 'pearson' in remove_test_positive[features].columns else []
    )

    test_positive_pool = Pool(X_test_positive, cat_features=categorical_features)

    for run in range(N_RUNS):
        negative_sampled = negative.sample(n=n_negative, random_state=421 + run)

        all_df = pd.concat([remove_test_positive, negative_sampled], axis=0).reset_index(drop=True)
        X_all = all_df[features]
        y_all = all_df['label']

        train_pool = Pool(X_all, y_all, cat_features=categorical_features)

        model = CatBoostClassifier(
            iterations=ITERATIONS,
            learning_rate=LEARNING_RATE,
            depth=DEPTH,
            loss_function=LOSS_FUNCTION,
            eval_metric=EVAL_METRIC,
            random_seed=RANDOM_SEED,
            verbose=VERBOSE
        )
        model.fit(train_pool)

        y_pred_positive = model.predict(test_positive_pool)
        recall = recall_score(y_test_positive, y_pred_positive)

        all_models.append({
            "year": YEAR,
            "run": run,
            "model": model,
            "recall": float(recall)
        })

    print(f"YEAR={YEAR} finished, {len([m for m in all_models if m['year']==YEAR])} models trained.")

In [ ]:
df_models = pd.DataFrame(all_models)

top50_models_per_year = (
    df_models.sort_values(["year", "recall"], ascending=[True, False])
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)

In [ ]:
ensemble_results = []

for year in range(2015, 2026):
    pos_after = positive[positive['formula'].isin(
        nanozyme_year[nanozyme_year['year'] == year].formula.tolist()
    )].reset_index(drop=True)

    X_pos_after = pos_after[features]

    categorical_features = (
        ['pearson'] if 'pearson' in X_pos_after.columns else []
    )
    pos_after_pool = Pool(X_pos_after, cat_features=categorical_features)

    subset = top50_models_per_year[top50_models_per_year["year"] == year]

    prob_list = []
    for _, row in subset.iterrows():
        model = row["model"]
        y_prob = model.predict_proba(pos_after_pool)[:, 1]
        prob_list.append(y_prob)

    avg_prob = np.mean(prob_list, axis=0)

    y_pred = (avg_prob >= 0.5).astype(int)

    total_samples = len(avg_prob)
    recalled_samples = y_pred.sum()
    recall = recalled_samples / total_samples

    ensemble_results.append({
        "year": year,
        "recalled_samples": int(recalled_samples),
        "unrecall_samples": int(total_samples - recalled_samples),
    })

ensemble_results_df = pd.DataFrame(ensemble_results)

In [ ]:
ensemble_results_df

In [ ]:
X_pos = positive[features].reset_index(drop=True)
categorical_features = (['pearson'] if 'pearson' in X_pos.columns else [])
pos = Pool(X_pos, cat_features=categorical_features)

row_index = positive['formula'].reset_index(drop=True)
col_dict = {}

for year in range(2015, 2026):
    subset = top50_models_per_year[top50_models_per_year["year"] == year]

    prob_list = []
    for _, r in subset.iterrows():
        model = r["model"]
        y_prob = model.predict_proba(pos)[:, 1]
        prob_list.append(y_prob)

    avg_prob = np.mean(np.vstack(prob_list), axis=0)

    col_dict[year] = avg_prob

df_prob = pd.DataFrame(col_dict, index=row_index)
df_prob[:] = (df_prob >= 0.5).astype('uint8')
df_prob = df_prob.merge(nanozyme_year[['formula', 'year']], on='formula', how='left')

In [ ]:
year_cols = list(range(2015, 2026))

hit_years = pd.DataFrame({y: np.where(df_prob[y] == 1, y, np.nan) for y in year_cols})
first_pred_year = hit_years.min(axis=1)

df_prob["first_pred_year"] = pd.to_numeric(first_pred_year, errors="coerce").astype("Int64")
df_prob["year_gap"] = (df_prob["year"] - df_prob["first_pred_year"]).astype("Int64")

In [ ]:
df_prob[df_prob["year_gap"]  >= 0].year_gap.value_counts().sort_index()